# Customer Churn Prediction & Revenue Recovery System

This notebook creates a full end-to-end churn analytics workflow for a B2B SaaS subscription business. It generates a synthetic dataset, explores churn patterns, builds predictive models, and quantifies ROI for retention interventions.


## 1. Setup & Imports


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from src.churn_utils import add_feature_engineering, generate_synthetic_churn_data, validate_dataset


## 2. Data Generation
We create 24 months of synthetic data with realistic churn decline behavior 2-3 months before churn.


In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

df = generate_synthetic_churn_data()
df.to_csv(DATA_DIR / 'churn_dataset.csv', index=False)

validate_dataset(df)
df.head()


## 3. Exploratory Data Analysis
Key churn patterns by segment, usage trends, and churn rate over time.


In [ ]:
df['month'] = pd.to_datetime(df['month'])
monthly_churn = df.groupby('month')['churned'].apply(lambda x: (x == 'Yes').mean()).reset_index()

plt.figure(figsize=(10, 4))
sns.lineplot(data=monthly_churn, x='month', y='churned')
plt.title('Monthly Churn Rate')
plt.ylabel('Churn Rate')
plt.tight_layout()
plt.show()


**Example Output:**
![Churn Rate Trend](../assets/churn_rate_trend.png)


## 4. Feature Engineering
We add rolling trends, recency metrics, engagement and health scores.


In [ ]:
df = add_feature_engineering(df)
df[['engagement_score', 'health_score']].describe()


## 5. Modeling (Logistic Regression, Random Forest, XGBoost)
We apply class weighting to address imbalance and evaluate using AUC, precision, recall, and F1.


In [ ]:
df['churn_flag'] = (df['churned'] == 'Yes').astype(int)
recent_df = df[df['month'] >= df['month'].max() - pd.DateOffset(months=5)].copy()

feature_cols = [
    'company_size', 'industry', 'region', 'subscription_tier',
    'contract_length', 'mrr', 'monthly_active_users', 'feature_adoption_rate',
    'login_frequency', 'api_calls', 'payment_delay', 'failed_payments',
    'support_tickets', 'support_response_time', 'support_satisfaction',
    'feature_requests', 'email_opens', 'webinar_attendance', 'documentation_views',
    'monthly_active_users_3m_trend', 'feature_adoption_rate_3m_trend',
    'login_frequency_3m_trend', 'api_calls_3m_trend', 'email_opens_3m_trend',
    'documentation_views_3m_trend', 'recency_days', 'engagement_score', 'health_score'
]

X = recent_df[feature_cols]
y = recent_df['churn_flag']

categorical_features = ['company_size', 'industry', 'region', 'subscription_tier']
numeric_features = [c for c in feature_cols if c not in categorical_features]

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', 'passthrough', numeric_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=250, max_depth=8, class_weight='balanced'),
    'XGBoost': XGBClassifier(
        n_estimators=250, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9, eval_metric='logloss'
    ),
}

metrics = {}
for name, model in models.items():
    clf = Pipeline([('preprocess', preprocess), ('model', model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    metrics[name] = {
        'auc': roc_auc_score(y_test, y_prob),
        'precision': precision_recall_fscore_support(y_test, y_pred, average='binary')[0],
        'recall': precision_recall_fscore_support(y_test, y_pred, average='binary')[1],
        'f1': precision_recall_fscore_support(y_test, y_pred, average='binary')[2],
    }

metrics


## 6. Model Evaluation
Confusion matrix and AUC-ROC for the best model.


In [ ]:
best_model = Pipeline([('preprocess', preprocess), ('model', models['XGBoost'])])
best_model.fit(X_train, y_train)

ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test)
plt.title('XGBoost Confusion Matrix')
plt.show()


**Example Output:**
![Confusion Matrix](../assets/xgb_confusion_matrix.png)


## 7. Feature Importance
Identify the top churn drivers to inform interventions.


In [ ]:
feature_importance = best_model.named_steps['model'].feature_importances_
feature_names = best_model.named_steps['preprocess'].get_feature_names_out()
importance_df = (
    pd.DataFrame({'feature': feature_names, 'importance': feature_importance})
    .sort_values('importance', ascending=False)
    .head(12)
)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x='importance', y='feature')
plt.title('Top Churn Drivers')
plt.tight_layout()
plt.show()


**Example Output:**
![Feature Importance](../assets/feature_importance.png)


## 8. Customer Segmentation (Risk × Value)


In [ ]:
latest_month = df['month'].max()
latest = df[df['month'] == latest_month].copy()
latest['churn_risk'] = best_model.predict_proba(latest[feature_cols])[:, 1]
latest['customer_value'] = pd.qcut(latest['mrr'], 2, labels=['Low', 'High'])
latest['risk_segment'] = pd.cut(latest['churn_risk'], [0, 0.5, 1.0], labels=['Low', 'High'])
latest['segment'] = latest['risk_segment'].astype(str) + ' Risk / ' + latest['customer_value'].astype(str) + ' Value'
latest['segment'].value_counts()


## 9. ROI Calculator & Backtesting
We estimate intervention costs, retention uplift, and 12-month revenue impact.


In [ ]:
segment_cost = {
    'High Risk / High Value': 500,
    'Low Risk / High Value': 50,
    'High Risk / Low Value': 10,
    'Low Risk / Low Value': 0,
}
segment_retention = {
    'High Risk / High Value': 0.25,
    'Low Risk / High Value': 0.08,
    'High Risk / Low Value': 0.12,
    'Low Risk / Low Value': 0.0,
}

latest['segment_key'] = latest['segment']
latest['intervention_cost'] = latest['segment_key'].map(segment_cost)
latest['retention_uplift'] = latest['segment_key'].map(segment_retention)
latest['expected_recovered_mrr'] = latest['mrr'] * latest['retention_uplift']

roi_summary = latest.groupby('segment_key').agg(
    customers=('customer_id', 'count'),
    total_mrr=('mrr', 'sum'),
    intervention_cost=('intervention_cost', 'sum'),
    expected_recovered_mrr=('expected_recovered_mrr', 'sum'),
).reset_index()
roi_summary['annual_recovered_revenue'] = roi_summary['expected_recovered_mrr'] * 12
roi_summary['roi'] = (roi_summary['annual_recovered_revenue'] - roi_summary['intervention_cost']) / roi_summary['intervention_cost'].replace(0, np.nan)

roi_summary


## 10. Save Artifacts
Export datasets, model metrics, and high-risk customer list for the dashboard.


In [ ]:
OUTPUT_DIR = Path('models')
OUTPUT_DIR.mkdir(exist_ok=True)

with open(OUTPUT_DIR / 'model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

latest[['customer_id', 'company_size', 'industry', 'region', 'subscription_tier', 'mrr', 'churn_risk', 'segment']]
.to_csv('data/high_risk_customers.csv', index=False)
roi_summary.to_csv('data/roi_summary.csv', index=False)
